<a href="https://colab.research.google.com/github/Cpea8713/IS4487/blob/main/assignment_12_bayes_svm_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv"
df = pd.read_csv(url)
print("Rows and columns:", df.shape)
print(df.dtypes)
features = ['hotel', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'market_segment', 'deposit_type', 'is_repeated_guest', 'previous_cancellations', 'booking_changes', 'adr']
X = df[features]
y = df['is_canceled']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_train.median(numeric_only=True))
X_train = pd.get_dummies(X_train, dummy_na=True)
X_test = pd.get_dummies(X_test, dummy_na=True).reindex(columns=X_train.columns, fill_value=0)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Rows and columns: (9404, 32)
hotel                              object
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                 object
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                            int64
babies                              int64
meal                               object
country                            object
market_segment                     object
distribution_channel               object
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                 object
assigned_room_type                 object
booking_changes                     int64
deposit_type                       object
agent

### ✍️ Your Response: 🔧
1. The dataset has 9,404 rows and 32 columns.

2. The dataset has both numerical and categorical features. Numerical features include lead time, number of adults, and number of nights. Categorical features include hotel type, market segment, and deposit type.

3. I selected 12 features to use for the models, filled in missing numerical values, and converted categorical features into numbers. I also split the data into 70% for training and 30% for testing.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Classification Report:\n", classification_report(y_test, y_pred_nb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

Accuracy: 0.8284904323175053
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.95      0.89      2111
           1       0.77      0.46      0.57       711

    accuracy                           0.83      2822
   macro avg       0.80      0.71      0.73      2822
weighted avg       0.82      0.83      0.81      2822

Confusion Matrix:
 [[2011  100]
 [ 384  327]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. The model had 82.85% accuracy, which seems pretty good. However, it only identified 46% of actual cancellations. I think recall is important because the hotel wants to catch as many potential cancellations as possible.

2. The hotel could use this model to identify bookings that might get canceled. This could help them plan staffing, prepare for empty rooms, and possibly reach out to customers who might cancel.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
svm_model = SVC(kernel='linear')
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Classification Report:\n", classification_report(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))

Accuracy: 0.8493975903614458
Classification Report:
               precision    recall  f1-score   support

           0       0.83      1.00      0.91      2111
           1       0.97      0.41      0.58       711

    accuracy                           0.85      2822
   macro avg       0.90      0.71      0.74      2822
weighted avg       0.87      0.85      0.83      2822

Confusion Matrix:
 [[2102    9]
 [ 416  295]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. The SVM model had 84.94% accuracy, which was slightly better than Naive Bayes. I think recall is also important because the hotel wants to identify customers who might cancel. The model only caught 41% of cancellations.

2. SVM could be useful when there are more complicated patterns in customer bookings. It could help the hotel predict cancellations based on several booking details instead of looking at just one or two factors.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
X_nn_train, X_val, y_nn_train, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42)
nn_model = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=200, random_state=42)
nn_model.fit(X_nn_train, y_nn_train)
y_pred_nn = nn_model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Classification Report:\n", classification_report(y_test, y_pred_nn))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nn))

Accuracy: 0.8607370659107016
Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.98      0.91      2111
           1       0.88      0.52      0.65       711

    accuracy                           0.86      2822
   macro avg       0.87      0.75      0.78      2822
weighted avg       0.86      0.86      0.85      2822

Confusion Matrix:
 [[2060   51]
 [ 342  369]]


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. The neural network performed the best with 86.07% accuracy. It also identified 52% of actual cancellations, which was better than both Naive Bayes and SVM.

2. One advantage is that neural networks can find more complicated patterns in the data. A disadvantage is that they can take longer to train and can be harder to understand compared to simpler models.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [ ]:
print("Naive Bayes:", accuracy_score(y_test, y_pred_nb))
print("SVM:", accuracy_score(y_test, y_pred_svm))
print("Neural Network:", accuracy_score(y_test, y_pred_nn))

Naive Bayes: 0.8284904323175053
SVM: 0.8493975903614458
Neural Network: 0.8607370659107016


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. The neural network had the best accuracy at 86.07%. Naive Bayes was the simplest and easiest to understand, while SVM was somewhere in between. We didn't measure exact training times, but the neural network was more complicated to train.

2. I would recommend the neural network because it had the highest accuracy and was better at identifying cancellations. However, I would want to test it more before actually using it since it still missed quite a few cancellations.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I would recommend using the neural network model because it had the highest accuracy at 86.07%. It could help the hotel predict cancellations and make better decisions about staffing and room availability. However, the model still missed almost half of the actual cancellations, so I wouldn't rely on it completely. I think having more information about customer booking habits and past cancellations could help improve its predictions.

2. My learning outcome is about using data to make better decisions in my house flipping business. Even though this assignment focuses on hotel cancellations, the same ideas could help me predict property values and returns on investments. It helped me understand how different models can be tested to see which gives the most accurate predictions, which could be useful when deciding whether a house is worth investing in.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 321097 bytes to assignment_12_bayes_svm_neural.html
